[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiptos/is_notes/blob/main/lstm.ipynb)

# Text Classification using LSTM

In this notebook, we classify news articles of the [AG News](https://huggingface.co/datasets/fancyzhx/ag_news) dataset into 4 topics: World, Sports, Business and Sci/Tech.

## Import libraries

First, it is necessary to import the required libraries. The dataset is downloaded with the Hugging Face `datasets` library (already installed in Colab; otherwise, install it with `!pip install datasets`).

In [ ]:
import re
from collections import Counter
from functools import partial

import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from datasets import load_dataset

tqdm.pandas()

## Download the dataset

The dataset has 120000 news articles (title and description) for training and 7600 for testing. We use the 120000 training articles, and split them ourselves into train, validation and test sets.

In [ ]:
ag_news = load_dataset("fancyzhx/ag_news")
print(ag_news)

# names of the classes (topics)
unique = ag_news['train'].features['label'].names
num_classes = len(unique)
print(unique)

sample = ag_news['train'].to_pandas()
sample

Check the number of articles of each class. In this dataset, the classes are balanced.

In [ ]:
sample['label'].map(lambda label: unique[label]).value_counts().plot(kind='bar', title='Number of articles in each topic')
plt.show()

As an example, let's print two records.

In [ ]:
def print_example(index):
    print(sample.text[index])
    print('Topic:', unique[sample.label[index]])


print_example(10)
print("-----------")
print_example(88)

# LSTM Model

Now we can start working towards the design and development of an LSTM model for classification.

Let's start by creating the `Dataset` class able to interpret, clean and convert the text. Note that the texts have some HTML leftovers, such as `#39;` (an apostrophe), `&lt;b&gt;` or `\` (a line break), which must be removed.

In [ ]:
import html
import nltk
from nltk.corpus import stopwords
from nltk import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
ENTITY_RE = re.compile(r'(?<!&)#(\d+);')  # entities without the '&', e.g. #39; for '
TAG_RE = re.compile(r'<[^>]+>')
REPLACE_BY_SPACE_RE = re.compile(r'[/(){}\[\]\|@,;\\]')
BAD_SYMBOLS_RE = re.compile(r'[^0-9a-z #+_]')
STOPWORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()


def clean_text(text):
    # convert HTML entities (e.g. &lt; or #39;) into characters and remove HTML tags
    text = html.unescape(ENTITY_RE.sub(r'&#\1;', text))
    text = TAG_RE.sub(' ', text)
    # lowercase text
    text = text.lower()
    # replace REPLACE_BY_SPACE_RE symbols by space in text. substitute the matched string in REPLACE_BY_SPACE_RE with space.
    text = REPLACE_BY_SPACE_RE.sub(' ', text)
    # remove symbols which are in BAD_SYMBOLS_RE from text. substitute the matched string in BAD_SYMBOLS_RE with nothing.
    text = BAD_SYMBOLS_RE.sub('', text)
    new_sent = ''
    for word in text.split():
        # Remove stop words
        if word not in STOPWORDS:
            # Lemmatize
            word = lemmatizer.lemmatize(word)
            new_sent = new_sent + ' ' + word
    text = new_sent
    return text


print(clean_text(sample.text[0]))
print(clean_text("HP: The Adaptive Enterprise that can't adapt &lt;strong&gt;Opinion&lt;/strong&gt; SAP hardly to blame"))

In [ ]:
def remove_rare_words(tokens, common_tokens, max_len):
    return [token if token in common_tokens
            else '<unk>' for token in tokens][-max_len:]


def tokenize(text, stop_words):
    # after clean_text, the text only has lowercase words separated by spaces
    tokens = text.split()
    tokens = [token for token in tokens if token not in stop_words]
    return tokens


class DataFrameTextDataset(Dataset):
    def __init__(self, text: list, label: list, max_vocab, max_len):
        df = pd.DataFrame(list(zip(text, label)), columns=['text', 'label'])

        # Clean and tokenize
        stop_words = set(stopwords.words('english'))
        df.text = df.text.progress_apply(clean_text)
        df.text = df.text.str.replace(r'\d+', '', regex=True)
        df['tokens'] = df.text.progress_apply(partial(tokenize, stop_words=stop_words))

        # Replace rare words with <unk>
        all_tokens = [sublst for lst in df.tokens.tolist() for sublst in lst]
        common_tokens = set(list(zip(*Counter(all_tokens).most_common(max_vocab)))[0])
        df.loc[:, 'tokens'] = df.tokens.progress_apply(
            partial(remove_rare_words, common_tokens=common_tokens, max_len=max_len))

        # Remove sequences with only <UNK>
        df = df[df.tokens.progress_apply(lambda tokens: any(token != '<unk>' for token in tokens))]

        vocab = sorted({sublst for lst in df.tokens.tolist() for sublst in lst})
        self.token2idx = {token: idx for idx, token in enumerate(vocab)}

        # Add a padding idx
        self.token2idx['<pad>'] = max(self.token2idx.values()) + 1

        self.idx2token = {idx: token for token, idx in self.token2idx.items()}

        df.loc[:, 'indexed_tokens'] = df.loc[:, 'tokens'].apply(
            lambda tokens: [self.token2idx[token] for token in tokens])
        self.text = df.text.tolist()
        self.sequences = df.indexed_tokens.tolist()
        self.targets = df.label.tolist()

    def __getitem__(self, i):
        return self.sequences[i], self.targets[i], self.text[i]

    def __len__(self):
        return len(self.sequences)

In [ ]:
# How many of the most common vocab words to keep
# Uncommon words get replaced with unknown token <unk>
max_vocab = 5000

# How many tokens long each sequence will be cut to
max_len = 100

dataset = DataFrameTextDataset(sample['text'].tolist(), sample['label'].tolist(), max_vocab, max_len)

Check the vocabulary...

In [ ]:
print(f'Vocabulary size: {len(dataset.token2idx)}')
list(dataset.token2idx.items())[:20]

Split dataset into train, validation, and test

In [ ]:
def split_train_valid_test(corpus, valid_ratio=0.1, test_ratio=0.1, generator=None):
    test_length = int(len(corpus) * test_ratio)
    valid_length = int(len(corpus) * valid_ratio)
    train_length = len(corpus) - valid_length - test_length
    return random_split(corpus, lengths=[train_length, valid_length, test_length], generator=generator)

In [ ]:
valid_ratio = 0.05
test_ratio = 0.05

generator = torch.Generator().manual_seed(42)
train_dataset, valid_dataset, test_dataset = split_train_valid_test(dataset, valid_ratio=valid_ratio, test_ratio=test_ratio,
                                                                    generator=generator)
len(train_dataset), len(valid_dataset), len(test_dataset)

Create the `DataLoader`s

In [ ]:
def collate(batch):
    inputs = [item[0] for item in batch]
    target = torch.LongTensor([item[1] for item in batch])
    text = [item[2] for item in batch]
    return inputs, target, text


batch_size = 512

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, collate_fn=collate)
test_loader = DataLoader(test_dataset, batch_size=batch_size, collate_fn=collate)

## Create the Model

The sequences have different lengths, so they are padded to the length of the longest one in the batch. Then, `pack_padded_sequence` tells the LSTM the real length of each sequence, so that the padding is ignored. The final hidden state of the LSTM (for a bidirectional LSTM, the final states of both directions) summarizes the text and is fed to the classifier.

Note that the model returns the raw scores (logits) of each class: `nn.CrossEntropyLoss` applies the softmax internally.

In [ ]:
class TextClassificationModel(nn.Module):
    def __init__(self, output_size, hidden_size, vocab_size, padding_idx,
                 device, dropout_probability=0.3, bidirectional=False, n_layers=1,
                 embedding_dimension=50):
        super().__init__()
        self.device = device
        self.padding_idx = padding_idx
        self.bidirectional = bidirectional

        # We need to multiply some layers by two if the model is bidirectional
        self.input_size_factor = 2 if bidirectional else 1

        self.embedding = nn.Embedding(vocab_size, embedding_dimension, padding_idx=padding_idx)

        self.rnn = nn.LSTM(embedding_dimension, hidden_size, n_layers, batch_first=True, bidirectional=bidirectional)
        self.drop = nn.Dropout(p=dropout_probability)
        self.fc1 = nn.Linear(hidden_size * self.input_size_factor, 16)
        self.fc2 = nn.Linear(16, output_size)

    def forward(self, inputs):
        lengths = torch.LongTensor([len(x) for x in inputs])

        # Pad sequences so that they are all the same length: (batch, max_length)
        padded_inputs = pad_sequence([torch.LongTensor(x) for x in inputs], batch_first=True,
                                     padding_value=self.padding_idx).to(self.device)
        # Get embeddings: (batch, max_length, embedding_dimension)
        embedding_out = self.embedding(padded_inputs)

        # The initial hidden and cell states are zeros (by default)
        packed = pack_padded_sequence(embedding_out, lengths, batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.rnn(packed)

        # h_n: (n_layers * directions, batch, hidden_size). Use the last layer.
        if self.bidirectional:
            hidden = torch.cat((h_n[-2], h_n[-1]), dim=1)  # forward and backward directions
        else:
            hidden = h_n[-1]

        x = self.drop(torch.relu(self.fc1(hidden)))
        return self.fc2(x)  # logits

Create the functions for training and validating. They return the average loss per example and the accuracy.

In [ ]:
def train_epoch(model, optimizer, scheduler, train_loader):
    model.train()
    total_loss = total = correct = 0
    progress_bar = tqdm(train_loader, desc='Training', leave=False)
    for inputs, target, text in progress_bar:
        target = target.to(device)
        # Clean old gradients
        optimizer.zero_grad()
        # Forwards pass
        output = model(inputs)
        # Calculate how wrong the model is
        loss = criterion(output, target)
        # Perform gradient descent, backwards pass
        loss.backward()
        # Take a step in the right direction
        optimizer.step()
        scheduler.step()
        # Record metrics (loss is the mean of the batch)
        total_loss += loss.item() * len(target)
        correct += (output.argmax(dim=1) == target).sum().item()
        total += len(target)
        progress_bar.set_postfix(loss=loss.item())

    return total_loss / total, correct / total


def validate_epoch(model, valid_loader):
    model.eval()
    total_loss = total = correct = 0
    with torch.no_grad():
        progress_bar = tqdm(valid_loader, desc='Validating', leave=False)
        for inputs, target, text in progress_bar:
            target = target.to(device)
            # Forwards pass
            output = model(inputs)
            # Calculate how wrong the model is
            loss = criterion(output, target)
            # Record metrics
            total_loss += loss.item() * len(target)
            correct += (output.argmax(dim=1) == target).sum().item()
            total += len(target)

    return total_loss / total, correct / total

## Finally, train the model

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
device

In [ ]:
dropout_probability = 0.2
n_rnn_layers = 1
embedding_dimension = 128
hidden_size = 64
is_bidirectional = True
max_epochs = 10
learning_rate = 0.001

model = TextClassificationModel(
    output_size=num_classes,
    hidden_size=hidden_size,
    embedding_dimension=embedding_dimension,
    vocab_size=len(dataset.token2idx),
    padding_idx=dataset.token2idx['<pad>'],
    dropout_probability=dropout_probability,
    bidirectional=is_bidirectional,
    n_layers=n_rnn_layers,
    device=device,
)
model = model.to(device)
model

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
# the learning rate decreases from learning_rate to 0 along the training (one step per batch)
scheduler = CosineAnnealingLR(optimizer, T_max=max_epochs * len(train_loader))

In [ ]:
n_epochs = 0
train_losses, valid_losses = [], []
for _ in range(max_epochs):
    train_loss, train_acc = train_epoch(model, optimizer, scheduler, train_loader)
    valid_loss, valid_acc = validate_epoch(model, valid_loader)

    tqdm.write(f'epoch #{n_epochs + 1:3d}\ttrain_loss: {train_loss:.4f}\ttrain_acc: {train_acc:.4f}'
               f'\tvalid_loss: {valid_loss:.4f}\tvalid_acc: {valid_acc:.4f}')

    # Early stopping if the current valid_loss is greater than the last three valid losses
    if len(valid_losses) > 2 and all(valid_loss >= loss for loss in valid_losses[-3:]):
        print('Stopping early')
        break

    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    n_epochs += 1

In [ ]:
epoch_ticks = range(1, n_epochs + 1)
plt.plot(epoch_ticks, train_losses)
plt.plot(epoch_ticks, valid_losses)
plt.legend(['Train Loss', 'Valid Loss'])
plt.title('Losses')
plt.xlabel('Epoch #')
plt.ylabel('Loss')
plt.xticks(epoch_ticks)
plt.show()

In [ ]:
torch.save(model.state_dict(), "lstm_model.pth")

In [ ]:
model = TextClassificationModel(
    output_size=num_classes,
    hidden_size=hidden_size,
    embedding_dimension=embedding_dimension,
    vocab_size=len(dataset.token2idx),
    padding_idx=dataset.token2idx['<pad>'],
    dropout_probability=dropout_probability,
    bidirectional=is_bidirectional,
    n_layers=n_rnn_layers,
    device=device,
)
model.load_state_dict(torch.load("lstm_model.pth", map_location=device, weights_only=True))
model = model.to(device)
model.eval()

## Evaluate the model

Precision, recall and F1-score of each class, with the test set.

In [ ]:
from sklearn.metrics import classification_report

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for inputs, target, text in tqdm(test_loader, leave=False):
        logits = model(inputs)
        predictions = logits.argmax(dim=1).cpu().numpy()

        y_true.extend(target.numpy())
        y_pred.extend(predictions)

print(classification_report(y_true, y_pred, labels=range(num_classes), target_names=list(unique), zero_division=0))

Let's see some random predictions.

In [ ]:
import html
import random
from IPython.display import HTML, display

flatten = lambda x: [sublst for lst in x for sublst in lst]
inputs_lst, target_lst, text_lst = zip(*test_loader)
inputs_lst, target_lst, text_lst = map(flatten, [inputs_lst, target_lst, text_lst])
test_examples = list(zip(inputs_lst, target_lst, text_lst))


def print_random_prediction(n=10):
    to_emoji = lambda y1, y2: '😄' if y1 == y2 else '😡'
    model.eval()
    rows = []
    for _ in range(n):
        with torch.no_grad():
            inputs, target, text = random.choice(test_examples)
            target = target.item()

            logits = model([inputs])
            prediction = logits.argmax(dim=1).item()

            predicted = to_emoji(target, prediction)

            row = f"""
            <tr>
            <td>{html.escape(text)}&nbsp;</td>
            <td>{unique[prediction]}&nbsp;</td>
            <td>{unique[target]}&nbsp;</td>
            <td>{predicted}&nbsp;</td>
            </tr>
            """
            rows.append(row)

    rows_joined = '\n'.join(rows)
    table = f"""
<table>
<tbody>
<tr>
<td><b>News</b>&nbsp;</td>
<td><b>Predicted</b>&nbsp;</td>
<td><b>Actual</b>&nbsp;</td>
<td><b>Correct?</b>&nbsp;</td>
</tr>
{rows_joined}
</tbody>
</table>
"""
    display(HTML(table))

In [ ]:
print_random_prediction(10)